In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-core/lra-core/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · The durable loop and its three invariants

Work through `core.py` by driving it. Fill in the `____` blanks; run each cell — the asserts tell you when you're right. Solutions are in `solutions/`.

In [ ]:
import sys; sys.path[:0] = ["..", "../.."]      # repo root, from notebooks/ or notebooks/solutions/
from core import Engine, Queue, Store, Crash, drain
from workflow import STEPS, CALLS
clock = [0.0]; now = lambda: clock[0]              # a clock we control: days pass in one line
def fresh(steps=STEPS):
    CALLS.clear(); return Engine(Store(), Queue(), dict(steps), clock=now, lease_ttl=60)


## 1. The run document

Start a run and look at the document. This is *all* the engine needs to resume the run on any machine, any time later.

In [ ]:
engine = fresh()
run = engine.start("run-1", "draft", {"topic": "durable agents"})
print(run)
print("queued:", engine.queue.tasks)          # (due, run_id, step, attempt)

## 2. Invariant I2 — a task is only executed if `(step, attempt)` matches

Deliver the first task by hand, then deliver it *again* (queues are at-least-once). Fill in what the second call must return.

In [ ]:
# TODO: fill in: dup, model_calls
print(engine.execute("run-1", "draft", 1))
print(engine.execute("run-1", "draft", 1))
assert engine.execute("run-1", "draft", 1) == ____
assert len([c for c in CALLS if c[0] == "model"]) == ____      # the model was called how many times?

## 3. Invariant I1 — checkpoint before enqueue

Read `Engine.execute` in `core.py`. Two writes happen at the end of a successful step. Put them in the right order and say what breaks in the other order.

In [ ]:
# TODO: fill in: first, second
ORDER = [____, ____]     # "checkpoint" or "enqueue"
WHY = "If we enqueue first and die before the checkpoint, the next task runs against state that was never saved (or is stale forever)."
assert ORDER == ["checkpoint", "enqueue"]

## 4. Invariant I3 — effects are recorded before the checkpoint

`publish` calls `ctx.effect(key, fn)`. Simulate the worst crash: the effect happened, then the process died before the checkpoint. On redelivery the effect must **not** run again.

In [ ]:
# TODO: fill in: seconds
engine = fresh()
engine.start("run-1", "draft", {"topic": "x"}); drain(engine, engine.queue, now)
engine.resume("run-1", "approval:run-1", {"decision": "approve"})

real_publish, died = engine.steps["publish"], []
def publish_then_die(ctx):
    out = real_publish(ctx)                 # cms_publish happened here
    if not died:
        died.append(1); raise Crash("died before the checkpoint")
    return out
engine.steps["publish"] = publish_then_die

print(drain(engine, engine.queue, now))
clock[0] += ____                      # wait for the dead worker's lease to expire
print(drain(engine, engine.queue, now))
publishes = [c for c in CALLS if c[0] == "publish"]
print("publish calls:", len(publishes), "| result:", engine.store.get("run-1")["result"])
assert len(publishes) == 1 and engine.store.get("run-1")["status"] == "SUCCEEDED"

## 5. Your turn: add an idempotent effect

Write a `bill` step that charges a card **at most once** even if the step is retried, then continues to `done_step`.

In [ ]:
# TODO: fill in: key
CHARGES = []
def bill(ctx):
    rec = ctx.effect(____, lambda: CHARGES.append(ctx.input["amount"]) or {"charge_id": "ch_1"})
    ctx.state["charge_id"] = rec["charge_id"]
    return ("next", "done_step")
def done_step(ctx):
    return ("done", ctx.state["charge_id"])

engine = fresh({"bill": bill, "done_step": done_step})
engine.start("inv-1", "bill", {"amount": 42})
engine.execute("inv-1", "bill", 1)
engine.execute("inv-1", "bill", 1)            # duplicate delivery
drain(engine, engine.queue, now)
assert CHARGES == [42] and engine.store.get("inv-1")["result"] == "ch_1"
print("charged exactly once:", CHARGES)